# Cluster and Visualize Unlabeled Acoustic Embeddings (PCA + HDBSCAN)

This notebook:
1. Creates interactive PCA plots for review (using first 3 components from 256D PCA)

### Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from os import environ

from sklearn.decomposition import PCA
import hdbscan
import plotly.express as px

In [ ]:
# Cell 2 — robust data path setup + load embeddings/manifest
env_dir = environ.get("POSIDONIA_DATASET_DIR")
DATASET_DIR = Path(env_dir) if env_dir else Path("D:/Posidonia Soundscapes/Fondeo 1_Formentera Ille Espardell/Embeddings_2/dataset")

EMBEDDINGS_PATH = DATASET_DIR / "npy_files" / "unlabeled_embeddings.npy"
MANIFEST_PATH = DATASET_DIR / "unlabeled_manifest.csv"

if not EMBEDDINGS_PATH.exists() or not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing files:\n"
        f"  embeddings: {EMBEDDINGS_PATH}\n"
        f"  manifest:   {MANIFEST_PATH}\n"
        f"Set POSIDONIA_DATASET_DIR to the correct dataset folder."
    )

embeddings = np.load(str(EMBEDDINGS_PATH))
manifest_df = pd.read_csv(str(MANIFEST_PATH))

print(f"Loaded {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")
print(f"Manifest rows: {len(manifest_df)}")
manifest_df.head()

In [ ]:
# Cell 3 — load or compute PCA (for visualization)
output_dir = EMBEDDINGS_PATH.parent
pca_output_dir = output_dir / "PCA_256D"
pca_output_dir.mkdir(parents=True, exist_ok=True)
pca_file = pca_output_dir / "pca_embeddings_256d.npy"

if pca_file.exists():
    print("Loading pre-computed PCA embeddings...")
    pca_results = np.load(str(pca_file))
    print(f"Loaded PCA: {pca_results.shape}")
else:
    print("Computing PCA(256D)...")
    embeddings_fp32 = embeddings.astype(np.float32, copy=False)
    pca_results = PCA(n_components=256, random_state=42).fit_transform(embeddings_fp32)

    np.save(str(pca_file), pca_results)
    print(f"Saved PCA embeddings to: {pca_file}")

print("PCA shape:", pca_results.shape)

In [ ]:
# Cell 4 — load or compute PCA cluster labels (for coloring in plots)
HDBSCAN_MIN_CLUSTER_SIZE = 80
HDBSCAN_MIN_SAMPLES = 10

pca_labels_file = pca_output_dir / f"pca_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"

if pca_labels_file.exists():
    print("Loading pre-computed PCA HDBSCAN labels...")
    pca_labels = np.load(str(pca_labels_file))
else:
    print("Computing HDBSCAN clustering on PCA embeddings...")
    pca_labels = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom"
    ).fit_predict(pca_results)
    np.save(str(pca_labels_file), pca_labels)

print("PCA labels:", pca_labels.shape)
print("Cluster distribution (includes -1 noise):")
print(pd.Series(pca_labels).value_counts().sort_index())

### Visualize

In [ ]:
# Cell 5 — static 3D PCA visualization (matplotlib)
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    pca_results[:, 0], pca_results[:, 1], pca_results[:, 2],
    c=pca_labels, cmap="gist_rainbow", s=5, alpha=0.3
)
ax.set_title("PCA HDBSCAN Clustering")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.set_zlabel("PCA 3")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — interactive PCA 3D (Plotly)
plot_df = pd.DataFrame({
    "x": pca_results[:, 0],
    "y": pca_results[:, 1],
    "z": pca_results[:, 2],
    "cluster": pca_labels.astype(str),
    "audio_file": manifest_df["audio_path"].apply(lambda x: Path(x).name) if "audio_path" in manifest_df.columns else "n/a"
})

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="cluster",
    hover_data=["cluster", "audio_file"],
    title="PCA (first 3 of 256 components) with HDBSCAN Clusters",
    labels={"x": "PCA 1", "y": "PCA 2", "z": "PCA 3"},
    opacity=0.7
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=800)
fig.show()